In [8]:
import requests
import pandas as pd

# Lista de todas as UFs brasileiras
estados = [
    'ac', 'al', 'ap', 'am', 'ba', 'ce', 'df', 'es', 'go', 'ma', 'mt', 'ms',
    'mg', 'pa', 'pb', 'pr', 'pe', 'pi', 'rj', 'rn', 'rs', 'ro', 'rr', 'sc',
    'sp', 'se', 'to'
]

resultados = []
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

print("Iniciando varredura em todos os estados no GitHub...\n")

for estado in estados:
    url_estado = f"https://api.github.com/repos/okfn-brasil/querido-diario/contents/data_collection/gazette/spiders/{estado}"

    response = requests.get(url_estado, headers=headers)

    if response.status_code == 200:
        dados_estado = response.json()

        spiders = [
            item['name'] for item in dados_estado
            if isinstance(item, dict) and item.get('type') == 'file' and item.get('name').endswith('.py') and item.get('name') != '__init__.py'
        ]

        # Guardamos a contagem por estado
        resultados.append({'Estado': estado.upper(), 'Total de Cidades': len(spiders)})

    elif response.status_code == 404:
        # Significa que a pasta do estado ainda não existe no repositório
        pass
    elif response.status_code == 403:
         print("\n[ALERTA] Limite de requisições do GitHub atingido (Rate Limit). Aguarde alguns minutos e tente novamente.")
         break

if resultados:
    df_brasil = pd.DataFrame(resultados)

    # Ordenando para mostrar os estados com mais cidades no topo
    df_brasil = df_brasil.sort_values(by='Total de Cidades', ascending=False).reset_index(drop=True)

    print("--- RANKING DE CIDADES MAPEADAS POR ESTADO ---")
    print(df_brasil.to_string())
    print(f"\nTotal geral de cidades com raspadores ativos: {df_brasil['Total de Cidades'].sum()}")

else:
    print("Nenhum dado pôde ser extraído. Verifique sua conexão ou limites de API.")

Iniciando varredura em todos os estados no GitHub...

--- RANKING DE CIDADES MAPEADAS POR ESTADO ---
   Estado  Total de Cidades
0      SP               115
1      BA                74
2      RJ                33
3      SE                29
4      MA                29
5      PR                26
6      TO                26
7      RS                21
8      MG                21
9      CE                13
10     PB                10
11     MS                10
12     RN                 8
13     PE                 6
14     AL                 4
15     AP                 3
16     ES                 3
17     MT                 3
18     SC                 3
19     PA                 3
20     GO                 2
21     DF                 1
22     AM                 1
23     PI                 1
24     RR                 1
25     RO                 1

Total geral de cidades com raspadores ativos: 447
